In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到停止销售时间，所以这里需要PLM的生命周期全表

In [71]:
month_date = 202512
channel = ['零售','工程','电商']
current_date = pd.Timestamp('2025-12-31')

productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}
productgroup_sort_map = {
    '吸油烟机': 0,
    '灶具': 1,
    '蒸烤微合计': 2,
    '灶集成': 3,
    '消毒柜': 4,
    '热水器': 5,
    '净水机': 6,
    '洗碗机': 7
}
pro_group_type_map={
    '吸油烟机': '吸油烟机',
    '灶具': '灶具',
    '烤箱': '蒸烤微合计',
    '蒸箱': '蒸烤微合计',
    '微波炉': '蒸烤微合计',
    '蒸烤烹饪机': '蒸烤微合计',
    '蒸烤微烹饪机': '蒸烤微合计',
    '蒸微': '蒸烤微合计',
    '灶消烹饪机': '灶集成',
    '灶蒸烹饪机': '灶集成',
    '灶蒸烤烹饪机': '灶集成',
    '灶烤烹饪机': '灶集成',
    '消毒柜': '消毒柜',
    '热水器': '热水器',
    '两用炉': '热水器',
    '家用净水机': '家用净水机',
    '商用净水机': '商用净水机',
    '水槽洗碗机': '水槽/嵌入洗碗机合计',
    '嵌入式洗碗机': '水槽/嵌入洗碗机合计'
}


### 读取物流数据，只保留3大渠道、并且是国内的数据

In [72]:
df = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx')
# df['物料号'] = df['商品编码'].astype(str).map(lambda x: x[:13])
# print(len(df))
df


,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价
0,1001001500116,零售,12,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358,40296
1,1009000600033,零售,1,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450,3450
2,1009000500035,零售,3,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280,15840
3,1001001500131,零售,6,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988,17928
4,1002003700049,零售,5,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550,12750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497614,1018001100016,零售,-12,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,在售,2025-06-19 12:00:00,NaT,3750,-45000
497615,1018001100017,电商,6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,NaN,NaT,NaT,4500,27000
497616,1018001100017,零售,-6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,在售,2025-08-22 12:00:00,NaT,4500,-27000
497617,1018001100018,电商,6,1018001100018,JBCD7E-04-M-Y1,嵌入式洗碗机,JBCD7E-04-M-Y1,国内,量产,NaN,NaT,NaT,4500,27000


In [73]:
# 长尾只看3大渠道。每个渠道的停止销售时间和既定时间的差距
df1 = df.copy()
df1 = df1[df1['对应渠道状态'] == '停止销售'].reset_index(drop=True)
df1 = df1[['物料编码','渠道','产品组','标准型号','国内/海外','对应渠道状态','产品状态','产品型号','停止销售时间']].drop_duplicates().reset_index(drop=True)
df1

,物料编码,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间
0,1001001500116,零售,吸油烟机,Z8T,国内,停止销售,停止销售,CXW-358-Z8T(不带罩),2025-11-24 16:51:46
1,1001001500131,零售,吸油烟机,02-Z6TA,国内,停止销售,停止销售,CXW-358-02-Z6TA(不带罩),2025-11-24 16:51:46
2,1002003700004,零售,灶具,02-HECB,国内,停止销售,停止销售,JZT-02-HE01CB-12T,2026-01-04 12:12:01
3,1001001500117,零售,吸油烟机,Z5TS,国内,停止销售,停止销售,CXW-358-Z5TS（不带罩）,2026-01-04 12:36:17
4,1004000500237,零售,热水器,JSQ25-H1303,国内,停止销售,量产,JSQ25-H1303-FR-12T,2025-11-14 15:02:45
...,...,...,...,...,...,...,...,...,...
453,1009000500020,电商,灶蒸烤烹饪机,JZT-ZK42-02-X3A.i,国内,停止销售,停止销售,JZT-ZK42-02-X3A.i,2025-06-09 11:01:32
454,1009000600001,电商,蒸烤烹饪机,ZK-T1.i,国内,停止销售,停止销售,ZK-T1.i,2025-05-27 11:13:44
455,1002003400143,工程,灶具,TH7B,国内,停止销售,停止销售,JZT-TH7B(SH)-12T,2025-12-25 15:12:36
456,1008000300023,电商,水槽洗碗机,JBSD2T-K3A,国内,停止销售,退市预警,JBSD2T-K3A,2025-07-23 15:02:58


### 保留各个渠道停止销售的产品

In [74]:
from calendar import month
from pandas import DateOffset
for index,row in df1.iterrows():
    if row['渠道'] == '零售':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df1.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '电商':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df1.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '工程':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=30):
            df1.loc[index,'是否长尾型号'] = '是'
df1


,物料编码,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间,是否长尾型号
0,1001001500116,零售,吸油烟机,Z8T,国内,停止销售,停止销售,CXW-358-Z8T(不带罩),2025-11-24 16:51:46,NaN
1,1001001500131,零售,吸油烟机,02-Z6TA,国内,停止销售,停止销售,CXW-358-02-Z6TA(不带罩),2025-11-24 16:51:46,NaN
2,1002003700004,零售,灶具,02-HECB,国内,停止销售,停止销售,JZT-02-HE01CB-12T,2026-01-04 12:12:01,NaN
3,1001001500117,零售,吸油烟机,Z5TS,国内,停止销售,停止销售,CXW-358-Z5TS（不带罩）,2026-01-04 12:36:17,NaN
4,1004000500237,零售,热水器,JSQ25-H1303,国内,停止销售,量产,JSQ25-H1303-FR-12T,2025-11-14 15:02:45,NaN
...,...,...,...,...,...,...,...,...,...,...
453,1009000500020,电商,灶蒸烤烹饪机,JZT-ZK42-02-X3A.i,国内,停止销售,停止销售,JZT-ZK42-02-X3A.i,2025-06-09 11:01:32,NaN
454,1009000600001,电商,蒸烤烹饪机,ZK-T1.i,国内,停止销售,停止销售,ZK-T1.i,2025-05-27 11:13:44,NaN
455,1002003400143,工程,灶具,TH7B,国内,停止销售,停止销售,JZT-TH7B(SH)-12T,2025-12-25 15:12:36,NaN
456,1008000300023,电商,水槽洗碗机,JBSD2T-K3A,国内,停止销售,退市预警,JBSD2T-K3A,2025-07-23 15:02:58,NaN


In [75]:
df1.to_excel(fr"C:\Users\zhangbon\Desktop\长尾明细.xlsx", index=False)


In [76]:
df2 = pd.merge(df,df1[['物料编码','渠道','是否长尾型号']],on=['物料编码','渠道'],how='left')
df2['是否长尾型号'] = df2['是否长尾型号'].fillna('否')
df2

,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价,是否长尾型号
0,1001001500116,零售,12,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358,40296,否
1,1009000600033,零售,1,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450,3450,否
2,1009000500035,零售,3,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280,15840,否
3,1001001500131,零售,6,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988,17928,否
4,1002003700049,零售,5,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550,12750,否
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497614,1018001100016,零售,-12,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,在售,2025-06-19 12:00:00,NaT,3750,-45000,否
497615,1018001100017,电商,6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,NaN,NaT,NaT,4500,27000,否
497616,1018001100017,零售,-6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,在售,2025-08-22 12:00:00,NaT,4500,-27000,否
497617,1018001100018,电商,6,1018001100018,JBCD7E-04-M-Y1,嵌入式洗碗机,JBCD7E-04-M-Y1,国内,量产,NaN,NaT,NaT,4500,27000,否


In [77]:
df_result = pd.DataFrame()
df_result['产品类别'] = productgroup_map.keys()
df_result
for k,v in productgroup_map.items():
    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')&(df2['渠道']=='零售')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'零售长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))&(df2['渠道']=='零售')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'零售产品型号数量'] = vals2
    df_result.loc[df_result['产品类别']==k,'零售长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')&(df2['渠道']=='工程')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'工程长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))&(df2['渠道']=='工程')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'工程产品型号数量'] = vals2
    df_result.loc[df_result['产品类别']==k,'工程长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')&(df2['渠道']=='电商')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'电商长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))&(df2['渠道']=='电商')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'电商产品型号数量'] = vals2 
    df_result.loc[df_result['产品类别']==k,'电商长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    

    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'全渠道长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'全渠道产品型号数量'] = vals2
    df_result.loc[df_result['产品类别']==k,'全渠道长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
df_result


,产品类别,零售长尾产品型号数量,零售产品型号数量,零售长尾产品型号占比,工程长尾产品型号数量,工程产品型号数量,工程长尾产品型号占比,电商长尾产品型号数量,电商产品型号数量,电商长尾产品型号占比,全渠道长尾产品型号数量,全渠道产品型号数量,全渠道长尾产品型号占比
0,吸油烟机,6.0,118.0,0.050847,21.0,108.0,0.194444,0.0,146.0,0.0,27.0,243.0,0.111111
1,灶具,0.0,215.0,0.000000,4.0,103.0,0.038835,0.0,197.0,0.0,4.0,344.0,0.011628
2,蒸烤微合计,0.0,50.0,0.000000,0.0,38.0,0.000000,0.0,57.0,0.0,0.0,70.0,0.000000
3,灶集成,0.0,68.0,0.000000,0.0,18.0,0.000000,0.0,30.0,0.0,0.0,72.0,0.000000
4,消毒柜,1.0,35.0,0.028571,0.0,27.0,0.000000,0.0,34.0,0.0,1.0,51.0,0.019608
5,热水器,0.0,66.0,0.000000,1.0,30.0,0.033333,0.0,57.0,0.0,1.0,92.0,0.010870
6,净水机,4.0,33.0,0.121212,0.0,18.0,0.000000,0.0,29.0,0.0,4.0,37.0,0.108108
7,洗碗机,3.0,81.0,0.037037,2.0,58.0,0.034483,0.0,119.0,0.0,5.0,150.0,0.033333


In [78]:
with pd.ExcelWriter(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\长尾统计结果-26新版.xlsx') as writer:
    df_result.to_excel(writer,sheet_name='分渠道产品型号统计',index=False)
